# Deep models: MLP and FT-Transformer, timed 5-fold CV

Run top to bottom with **Runtime → Restart and run all**. All settings live in the configuration cell.

In [1]:
%pip install -q rtdl_revisiting_models==0.0.2

## Βήμα 0: Configuration and reproducibility

In [2]:
import os

N_THREADS = 4
SEED = 123
N_FOLDS = 5
DEVICE = None
DATA_PATH = "/content/drive/MyDrive/Base.csv"
SAVE_DIR = "/content/drive/MyDrive/thesis_cv_results"
PDF_PATH = os.path.join(SAVE_DIR, "FINAL_Combined_Thesis_Report.pdf")

MLP_CFG = dict(epochs=30, patience=5, batch_size=2048, lr=1e-3, weight_decay=1e-5,
               hidden_dim=256, n_layers=3, dropout=0.2)
FTT_CFG = dict(epochs=60, patience=5, batch_size=4096, lr=1e-4, weight_decay=1e-5,
               eval_batch_size=8192)

for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = str(N_THREADS)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONHASHSEED"] = str(SEED)

In [3]:
import gc
import json
import pickle
import platform
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import torch
import torch.nn as nn
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, QuantileTransformer
from torch.amp import GradScaler, autocast
from rtdl_revisiting_models import FTTransformer

warnings.filterwarnings("ignore")

torch.set_num_threads(N_THREADS)
try:
    torch.set_num_interop_threads(N_THREADS)
except RuntimeError:
    pass

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device(DEVICE or ("cuda" if torch.cuda.is_available() else "cpu"))


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def sync(dev):
    if dev.type == "cuda":
        torch.cuda.synchronize()


set_seed(SEED)
ENV = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "sklearn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "device": DEVICE.type,
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "intra_op_threads": torch.get_num_threads(),
}
print(json.dumps(ENV, indent=2))

{
  "python": "3.13.15",
  "torch": "2.11.0+cu130",
  "sklearn": "1.6.1",
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "device": "cuda",
  "gpu": "Tesla T4",
  "intra_op_threads": 4
}


## Βήμα 1: Data loading and cleaning

In [4]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

os.makedirs(SAVE_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)

df = df.drop(columns=["device_fraud_count", "prev_address_months_count",
                      "intended_balcon_amount", "bank_months_count"], errors="ignore")
for col in ["current_address_months_count", "session_length_in_minutes",
            "device_distinct_emails_8w"]:
    df.loc[df[col] == -1, col] = np.nan
df = df.dropna().reset_index(drop=True)
df["fraud_bool"] = df["fraud_bool"].astype(int)

cols_to_factor = ["email_is_free", "phone_home_valid", "phone_mobile_valid",
                  "has_other_cards", "foreign_request", "keep_alive_session",
                  "month", "payment_type", "employment_status",
                  "housing_status", "source", "device_os",
                  "device_distinct_emails_8w"]
num_cols_nn = [c for c in df.columns if c not in cols_to_factor and c != "fraud_bool"]

print(f"Rows after cleaning: {len(df):,} | fraud rate: {df['fraud_bool'].mean():.3%}")
print(f"Numerical ({len(num_cols_nn)}): {num_cols_nn}")
print(f"Categorical ({len(cols_to_factor)}):")
print(df[cols_to_factor].nunique().sort_values())

Mounted at /content/drive
Rows after cleaning: 993,607 | fraud rate: 1.107%
Numerical (14): ['income', 'name_email_similarity', 'current_address_months_count', 'customer_age', 'days_since_request', 'zip_count_4w', 'velocity_6h', 'velocity_24h', 'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w', 'credit_risk_score', 'proposed_credit_limit', 'session_length_in_minutes']
Categorical (13):
email_is_free                2
phone_home_valid             2
phone_mobile_valid           2
has_other_cards              2
foreign_request              2
keep_alive_session           2
source                       2
device_distinct_emails_8w    3
device_os                    5
payment_type                 5
employment_status            7
housing_status               7
month                        8
dtype: int64


## Βήμα 2: Held-out test set and CV folds

In [5]:
X_nn = df.drop(columns=["fraud_bool"])
y_nn = df["fraud_bool"]

X_tr_nn, X_tmp, y_tr_nn, y_tmp = train_test_split(
    X_nn, y_nn, test_size=0.30, stratify=y_nn, random_state=SEED)
X_val_nn, X_te_nn, y_val_nn, y_te_nn = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

X_cv_nn = pd.concat([X_tr_nn, X_val_nn]).reset_index(drop=True)
y_cv_nn = pd.concat([y_tr_nn, y_val_nn]).reset_index(drop=True)
X_te_nn = X_te_nn.reset_index(drop=True)
y_te_nn = y_te_nn.reset_index(drop=True)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X_cv_nn, y_cv_nn))

le_dict = {}
for col in cols_to_factor:
    le = LabelEncoder()
    le.fit(df[col].astype(str))
    le_dict[col] = le
cat_cardinalities = [len(le_dict[c].classes_) for c in cols_to_factor]

print(f"CV pool : {len(X_cv_nn):>9,} ({y_cv_nn.mean():.3%} fraud)")
print(f"Test set: {len(X_te_nn):>9,} ({y_te_nn.mean():.3%} fraud)")
print(f"cat_cardinalities: {cat_cardinalities}")

CV pool :   844,565 (1.106% fraud)
Test set:   149,042 (1.107% fraud)
cat_cardinalities: [2, 2, 2, 2, 2, 2, 8, 5, 7, 7, 2, 5, 3]


## Βήμα 3: Fold preprocessing (fit on the fold's training split only)

In [6]:
def label_encode(X):
    out = np.zeros((len(X), len(cols_to_factor)), dtype=np.int64)
    for j, col in enumerate(cols_to_factor):
        out[:, j] = le_dict[col].transform(X[col].astype(str))
    return out


def prepare_fold(train_idx, val_idx):
    X_f_tr = X_cv_nn.iloc[train_idx].reset_index(drop=True)
    X_f_val = X_cv_nn.iloc[val_idx].reset_index(drop=True)
    y_f_tr = y_cv_nn.iloc[train_idx].reset_index(drop=True)
    y_f_val = y_cv_nn.iloc[val_idx].reset_index(drop=True)

    qt = QuantileTransformer(output_distribution="normal", random_state=SEED)
    num_tr = qt.fit_transform(X_f_tr[num_cols_nn]).astype(np.float32)
    num_val = qt.transform(X_f_val[num_cols_nn]).astype(np.float32)
    num_te = qt.transform(X_te_nn[num_cols_nn]).astype(np.float32)

    ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore", drop="first")
    ohe_tr = ohe.fit_transform(X_f_tr[cols_to_factor].astype(str)).astype(np.float32)
    ohe_val = ohe.transform(X_f_val[cols_to_factor].astype(str)).astype(np.float32)
    ohe_te = ohe.transform(X_te_nn[cols_to_factor].astype(str)).astype(np.float32)

    n_neg, n_pos = int((y_f_tr == 0).sum()), int((y_f_tr == 1).sum())
    return {
        "y_tr": y_f_tr.values.astype(np.float32),
        "y_val": y_f_val.values.astype(np.float32),
        "mlp_tr": np.hstack([num_tr, ohe_tr]),
        "mlp_val": np.hstack([num_val, ohe_val]),
        "mlp_te": np.hstack([num_te, ohe_te]),
        "num_tr": num_tr, "num_val": num_val, "num_te": num_te,
        "cat_tr": label_encode(X_f_tr), "cat_val": label_encode(X_f_val),
        "cat_te": label_encode(X_te_nn),
        "pos_weight": torch.tensor([n_neg / n_pos], dtype=torch.float32),
    }

## Βήμα 4: Models and the shared timed training loop

MLP: 3 hidden layers × 256 units · LayerNorm · GELU · Dropout 0.2.
FT-Transformer: rtdl defaults (`FTTransformer.get_default_kwargs()`).
Both: AdamW, cosine LR, `BCEWithLogitsLoss(pos_weight)`, early stopping on validation PR-AUC.
Fit time covers the full training loop including per-epoch validation; score time covers test-set inference only.

In [7]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, n_layers=3, dropout=0.2):
        super().__init__()
        layers, in_dim = [], input_dim
        for _ in range(n_layers):
            layers += [nn.Linear(in_dim, hidden_dim), nn.LayerNorm(hidden_dim),
                       nn.GELU(), nn.Dropout(dropout)]
            in_dim = hidden_dim
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def build_mlp(input_dim):
    model = MLP(input_dim, MLP_CFG["hidden_dim"], MLP_CFG["n_layers"], MLP_CFG["dropout"])
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            nn.init.zeros_(m.bias)
    return model


def build_ftt(n_cont):
    kwargs = FTTransformer.get_default_kwargs()
    kwargs.pop("_is_default", None)
    return FTTransformer(n_cont_features=n_cont, cat_cardinalities=cat_cardinalities,
                         d_out=1, **kwargs)


def predict_logits(model, inputs, batch_size, dev, use_amp):
    n = len(inputs[0])
    out = []
    with torch.no_grad():
        for i in range(0, n, batch_size):
            xb = [t[i:i + batch_size] for t in inputs]
            with autocast(dev.type, enabled=use_amp):
                out.append(model(*xb).float())
    return torch.cat(out)


def train_eval(model, tr_inputs, y_tr, val_inputs, y_val, te_inputs, y_te,
               pos_weight, cfg, seed, dev, use_amp):
    eval_bs = cfg.get("eval_batch_size", 8192)
    to = lambda a, dt: torch.as_tensor(a, dtype=dt, device=dev)
    dtypes = [torch.float32 if a.dtype == np.float32 else torch.long for a in tr_inputs]
    tr = [to(a, d) for a, d in zip(tr_inputs, dtypes)]
    va = [to(a, d) for a, d in zip(val_inputs, dtypes)]
    te = [to(a, d) for a, d in zip(te_inputs, dtypes)]
    y_tr_t = to(y_tr, torch.float32).unsqueeze(1)
    y_val_t = to(y_val, torch.float32).unsqueeze(1)

    model = model.to(dev)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"],
                                  weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(dev))
    scaler = GradScaler("cuda") if use_amp else None
    gen = torch.Generator().manual_seed(seed)

    n, bs = len(y_tr), cfg["batch_size"]
    n_batches = (n + bs - 1) // bs
    best_pr, best_state, best_epoch, wait = -1.0, None, 0, 0
    history = {"train_loss": [], "val_loss": [], "val_pr_auc": []}

    sync(dev)
    t0 = time.perf_counter()
    for epoch in range(cfg["epochs"]):
        model.train()
        perm = torch.randperm(n, generator=gen).to(dev)
        total = 0.0
        for b in range(n_batches):
            idx = perm[b * bs:(b + 1) * bs]
            optimizer.zero_grad(set_to_none=True)
            with autocast(dev.type, enabled=use_amp):
                loss = criterion(model(*[t[idx] for t in tr]), y_tr_t[idx])
            if use_amp:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            total += loss.item()
        scheduler.step()

        model.eval()
        val_logits = predict_logits(model, va, eval_bs, dev, use_amp)
        val_loss = criterion(val_logits, y_val_t).item()
        val_pr = average_precision_score(y_val, torch.sigmoid(val_logits).cpu().numpy())
        history["train_loss"].append(total / n_batches)
        history["val_loss"].append(val_loss)
        history["val_pr_auc"].append(val_pr)

        if val_pr > best_pr:
            best_pr, best_epoch, wait = val_pr, epoch + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
        if (epoch + 1) % 5 == 0:
            print(f"    Epoch {epoch + 1:3d} | Train Loss: {total / n_batches:.4f} "
                  f"| Val Loss: {val_loss:.4f} | Val PR-AUC: {val_pr:.4f}")
        if wait >= cfg["patience"]:
            print(f"    Early stopping at epoch {epoch + 1}")
            break
    sync(dev)
    fit_seconds = time.perf_counter() - t0

    model.load_state_dict(best_state)
    model.eval()
    val_probs = torch.sigmoid(predict_logits(model, va, eval_bs, dev, use_amp)).cpu().numpy().ravel()

    sync(dev)
    t1 = time.perf_counter()
    test_probs = torch.sigmoid(predict_logits(model, te, eval_bs, dev, use_amp)).cpu().numpy().ravel()
    sync(dev)
    score_seconds = time.perf_counter() - t1

    return {
        "val_roc_auc": roc_auc_score(y_val, val_probs),
        "val_pr_auc": average_precision_score(y_val, val_probs),
        "test_roc_auc": roc_auc_score(y_te, test_probs),
        "test_pr_auc": average_precision_score(y_te, test_probs),
        "test_probs": test_probs,
        "history": history,
        "best_epoch": best_epoch,
        "epochs_run": len(history["val_pr_auc"]),
        "fit_seconds": fit_seconds,
        "score_seconds": score_seconds,
        "score_ms_per_1000": score_seconds * 1e6 / len(y_te),
        "n_test": len(y_te),
        "device": dev.type,
        "n_threads": torch.get_num_threads(),
        "state_dict": best_state,
    }

## Βήμα 5: Timed 5-fold CV for both models

In [8]:
def run_cv(name):
    results = []
    print("=" * 60)
    print(f"{name} — {N_FOLDS}-Fold Stratified CV on {DEVICE.type}")
    print("=" * 60)
    for fold_idx, (train_idx, val_idx) in enumerate(FOLDS):
        seed = 42 + fold_idx
        print(f"\n  Fold {fold_idx + 1}/{N_FOLDS} (seed {seed})")
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        gc.collect()

        d = prepare_fold(train_idx, val_idx)
        print(f"    Train {len(d['y_tr']):,} | Val {len(d['y_val']):,} "
              f"| pos_weight: {d['pos_weight'].item():.2f}")

        set_seed(seed)
        if name == "MLP":
            model = build_mlp(d["mlp_tr"].shape[1])
            res = train_eval(model, [d["mlp_tr"]], d["y_tr"], [d["mlp_val"]], d["y_val"],
                             [d["mlp_te"]], y_te_nn.values, d["pos_weight"],
                             MLP_CFG, seed, DEVICE, use_amp=False)
        else:
            model = build_ftt(len(num_cols_nn))
            res = train_eval(model, [d["num_tr"], d["cat_tr"]], d["y_tr"],
                             [d["num_val"], d["cat_val"]], d["y_val"],
                             [d["num_te"], d["cat_te"]], y_te_nn.values, d["pos_weight"],
                             FTT_CFG, seed, DEVICE, use_amp=(DEVICE.type == "cuda"))
        res["fold"], res["seed"] = fold_idx + 1, seed
        results.append(res)
        print(f"    Val  → ROC-AUC: {res['val_roc_auc']:.4f} | PR-AUC: {res['val_pr_auc']:.4f}")
        print(f"    Test → ROC-AUC: {res['test_roc_auc']:.4f} | PR-AUC: {res['test_pr_auc']:.4f}")
        print(f"    Fit {res['fit_seconds']:.1f}s | Score {res['score_seconds']:.3f}s "
              f"| {res['score_ms_per_1000']:.2f} ms/1000 rows")
        del model, d
    return results


cv_mlp_results = run_cv("MLP")

MLP — 5-Fold Stratified CV on cuda

  Fold 1/5 (seed 42)
    Train 675,652 | Val 168,913 | pos_weight: 89.38
    Epoch   5 | Train Loss: 0.8440 | Val Loss: 0.8600 | Val PR-AUC: 0.1469
    Epoch  10 | Train Loss: 0.8044 | Val Loss: 0.8629 | Val PR-AUC: 0.1484
    Epoch  15 | Train Loss: 0.7662 | Val Loss: 0.8735 | Val PR-AUC: 0.1466
    Early stopping at epoch 18
    Val  → ROC-AUC: 0.8800 | PR-AUC: 0.1499
    Test → ROC-AUC: 0.8740 | PR-AUC: 0.1438
    Fit 26.8s | Score 0.024s | 0.16 ms/1000 rows

  Fold 2/5 (seed 43)
    Train 675,652 | Val 168,913 | pos_weight: 89.38
    Epoch   5 | Train Loss: 0.8505 | Val Loss: 0.8517 | Val PR-AUC: 0.1390
    Epoch  10 | Train Loss: 0.8127 | Val Loss: 0.8435 | Val PR-AUC: 0.1426
    Epoch  15 | Train Loss: 0.7782 | Val Loss: 0.8862 | Val PR-AUC: 0.1421
    Early stopping at epoch 19
    Val  → ROC-AUC: 0.8857 | PR-AUC: 0.1428
    Test → ROC-AUC: 0.8740 | PR-AUC: 0.1448
    Fit 27.5s | Score 0.026s | 0.17 ms/1000 rows

  Fold 3/5 (seed 44)
    Train

In [9]:
cv_ftt_results = run_cv("FT-Transformer")

FT-Transformer — 5-Fold Stratified CV on cuda

  Fold 1/5 (seed 42)
    Train 675,652 | Val 168,913 | pos_weight: 89.38
    Epoch   5 | Train Loss: 0.8147 | Val Loss: 0.8226 | Val PR-AUC: 0.1640
    Epoch  10 | Train Loss: 0.8016 | Val Loss: 0.8188 | Val PR-AUC: 0.1594
    Epoch  15 | Train Loss: 0.7886 | Val Loss: 0.8234 | Val PR-AUC: 0.1658
    Early stopping at epoch 18
    Val  → ROC-AUC: 0.8934 | PR-AUC: 0.1690
    Test → ROC-AUC: 0.8877 | PR-AUC: 0.1651
    Fit 587.9s | Score 2.254s | 15.13 ms/1000 rows

  Fold 2/5 (seed 43)
    Train 675,652 | Val 168,913 | pos_weight: 89.38
    Epoch   5 | Train Loss: 0.8178 | Val Loss: 0.8112 | Val PR-AUC: 0.1635
    Epoch  10 | Train Loss: 0.8028 | Val Loss: 0.8079 | Val PR-AUC: 0.1653
    Epoch  15 | Train Loss: 0.7845 | Val Loss: 0.8348 | Val PR-AUC: 0.1642
    Early stopping at epoch 16
    Val  → ROC-AUC: 0.8947 | PR-AUC: 0.1665
    Test → ROC-AUC: 0.8876 | PR-AUC: 0.1664
    Fit 523.1s | Score 2.254s | 15.12 ms/1000 rows

  Fold 3/5 (see

## Βήμα 6: Metrics (threshold 0.5) and timings, mean ± std across folds

In [10]:
def threshold_metrics(probs, thr=0.5):
    tn, fp, fn, tp = confusion_matrix(y_te_nn, (probs >= thr).astype(int)).ravel()
    return {
        "Accuracy": (tp + tn) / (tp + tn + fp + fn),
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "Precision": tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        "F1-Score": 2 * tp / (2 * tp + fp + fn),
    }


def per_fold_table(results, name):
    rows = []
    for r in results:
        row = {"Model": name, "Fold": r["fold"], "Seed": r["seed"]}
        row.update(threshold_metrics(r["test_probs"]))
        row.update({"ROC-AUC": r["test_roc_auc"], "PR-AUC": r["test_pr_auc"],
                    "Val ROC-AUC": r["val_roc_auc"], "Val PR-AUC": r["val_pr_auc"],
                    "Best epoch": r["best_epoch"], "Epochs run": r["epochs_run"],
                    "Fit (s)": r["fit_seconds"], "Score (s)": r["score_seconds"],
                    "Score (ms/1000)": r["score_ms_per_1000"],
                    "Device": r["device"], "Threads": r["n_threads"]})
        rows.append(row)
    return pd.DataFrame(rows)


per_fold = pd.concat([per_fold_table(cv_mlp_results, "MLP"),
                      per_fold_table(cv_ftt_results, "FT-Transformer")], ignore_index=True)

metric_cols = ["Accuracy", "Sensitivity", "Specificity", "Precision", "F1-Score",
               "ROC-AUC", "PR-AUC"]
timing_cols = ["Fit (s)", "Score (s)", "Score (ms/1000)"]
fmt = lambda s, d: f"{s.mean():.{d}f} +/- {s.std(ddof=0):.{d}f}"

summary = pd.DataFrame([
    {"Model": f"{m} ({N_FOLDS}-Fold CV)", **{c: fmt(g[c], 4) for c in metric_cols}}
    for m, g in per_fold.groupby("Model", sort=False)])
timings = pd.DataFrame([
    {"Model": m, "Fit (s)": fmt(g["Fit (s)"], 1), "Score (s)": fmt(g["Score (s)"], 3),
     "Score (ms/1000)": fmt(g["Score (ms/1000)"], 2)}
    for m, g in per_fold.groupby("Model", sort=False)])

pd.set_option("display.width", 200)
print(summary.to_string(index=False))
print(f"\nTimings (device={DEVICE.type}, threads={torch.get_num_threads()}, "
      f"n_test={len(y_te_nn):,})")
print(timings.to_string(index=False))

                     Model          Accuracy       Sensitivity       Specificity         Precision          F1-Score           ROC-AUC            PR-AUC
           MLP (5-Fold CV) 0.8285 +/- 0.0085 0.7565 +/- 0.0136 0.8293 +/- 0.0088 0.0473 +/- 0.0017 0.0891 +/- 0.0029 0.8750 +/- 0.0013 0.1422 +/- 0.0020
FT-Transformer (5-Fold CV) 0.8185 +/- 0.0284 0.7898 +/- 0.0376 0.8188 +/- 0.0291 0.0475 +/- 0.0065 0.0895 +/- 0.0111 0.8877 +/- 0.0007 0.1655 +/- 0.0020

Timings (device=cuda, threads=4, n_test=149,042)
         Model        Fit (s)       Score (s) Score (ms/1000)
           MLP   20.5 +/- 5.5 0.027 +/- 0.005   0.18 +/- 0.03
FT-Transformer 536.1 +/- 93.7 2.254 +/- 0.001  15.13 +/- 0.01


## Βήμα 7: PDF report

In [11]:
mlp_roc = [r["test_roc_auc"] for r in cv_mlp_results]
ftt_roc = [r["test_roc_auc"] for r in cv_ftt_results]
best_mlp_idx = int(np.argmax(mlp_roc))
best_ftt_idx = int(np.argmax(ftt_roc))


def table_page(pdf, frame, title, width):
    fig, ax = plt.subplots(figsize=(width, 3))
    ax.axis("off")
    t = ax.table(cellText=frame.values, colLabels=frame.columns, loc="center", cellLoc="center")
    t.auto_set_font_size(False)
    t.set_fontsize(9)
    t.scale(1, 2)
    ax.set_title(title, pad=20, fontsize=14, fontweight="bold")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


def curves_page(pdf, res, name, color):
    h = res["history"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(h["train_loss"], label="Train Loss", color="C0")
    axes[0].plot(h["val_loss"], label="Val Loss", color="C1")
    axes[0].set_title(f"{name}: Train vs Val Loss (Fold {res['fold']})")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[1].plot(h["val_pr_auc"], label="Val PR-AUC", color=color)
    axes[1].set_title(f"{name}: Validation PR-AUC")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("PR-AUC")
    for ax in axes:
        ax.legend(); ax.grid(True)
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


with PdfPages(PDF_PATH) as pdf:
    table_page(pdf, summary, f"Neural Network Models — {N_FOLDS}-Fold CV Results", 16)
    table_page(pdf, timings,
               f"Fit / scoring time (device={DEVICE.type}, threads={torch.get_num_threads()})", 10)
    curves_page(pdf, cv_mlp_results[best_mlp_idx], "MLP", "C2")
    curves_page(pdf, cv_ftt_results[best_ftt_idx], "FT-Transformer", "C3")

    fig = plt.figure(figsize=(7, 5))
    sns.boxplot(data=pd.DataFrame({"MLP": mlp_roc, "FT-Transformer": ftt_roc}), palette="Set2")
    plt.title(f"NN Models: Test ROC-AUC across {N_FOLDS} CV Folds")
    plt.ylabel("ROC-AUC"); plt.grid(axis="y"); plt.tight_layout()
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, res, name, cmap in [(axes[0], cv_mlp_results[best_mlp_idx], "MLP", "Blues"),
                                (axes[1], cv_ftt_results[best_ftt_idx], "FT-Transformer", "Greens")]:
        cm = confusion_matrix(y_te_nn, (res["test_probs"] >= 0.5).astype(int))
        sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=ax,
                    xticklabels=["Legit (0)", "Fraud (1)"], yticklabels=["Legit (0)", "Fraud (1)"])
        ax.set_title(f"{name} Confusion Matrix (Fold {res['fold']})")
        ax.set_ylabel("Actual"); ax.set_xlabel("Predicted")
    fig.tight_layout()
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

print(f"Report: {PDF_PATH}")

Report: /content/drive/MyDrive/thesis_cv_results/FINAL_Combined_Thesis_Report.pdf


## Βήμα 8: Save results

In [12]:
def save_results(results, prefix):
    light = []
    for r in results:
        torch.save(r["state_dict"], os.path.join(SAVE_DIR, f"{prefix}_fold{r['fold']}_weights.pt"))
        light.append({k: v for k, v in r.items() if k != "state_dict"})
    with open(os.path.join(SAVE_DIR, f"cv_{prefix}_results.pkl"), "wb") as f:
        pickle.dump(light, f)


save_results(cv_mlp_results, "mlp")
save_results(cv_ftt_results, "ftt")
y_te_nn.to_pickle(os.path.join(SAVE_DIR, "y_te_nn.pkl"))
per_fold.to_csv(os.path.join(SAVE_DIR, "cv_per_fold.csv"), index=False)
summary.to_csv(os.path.join(SAVE_DIR, "cv_summary.csv"), index=False)
timings.to_csv(os.path.join(SAVE_DIR, "cv_timings.csv"), index=False)
with open(os.path.join(SAVE_DIR, "run_config.json"), "w") as f:
    json.dump({"env": ENV, "seed": SEED, "n_folds": N_FOLDS, "fold_seeds": [42 + i for i in range(N_FOLDS)],
               "mlp": MLP_CFG, "ftt": FTT_CFG, "n_cv": len(y_cv_nn), "n_test": len(y_te_nn),
               "cat_cardinalities": cat_cardinalities}, f, indent=2)

for f in sorted(os.listdir(SAVE_DIR)):
    print(f"  {f}  ({os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024:.1f} KB)")

  FINAL_Combined_Thesis_Report.pdf  (41.9 KB)
  cv_ftt_results.pkl  (2915.4 KB)
  cv_mlp_results.pkl  (2915.0 KB)
  cv_per_fold.csv  (2.7 KB)
  cv_summary.csv  (0.4 KB)
  cv_timings.csv  (0.1 KB)
  final_ftt_result.pkl  (8733.5 KB)
  final_mlp_result.pkl  (8733.5 KB)
  ftt_fold1_weights.pt  (3572.7 KB)
  ftt_fold2_weights.pt  (3572.7 KB)
  ftt_fold3_weights.pt  (3572.7 KB)
  ftt_fold4_weights.pt  (3572.7 KB)
  ftt_fold5_weights.pt  (3572.7 KB)
  ftt_tuning  (4.0 KB)
  ldam_ablation  (4.0 KB)
  mlp_fold1_weights.pt  (577.1 KB)
  mlp_fold2_weights.pt  (577.1 KB)
  mlp_fold3_weights.pt  (577.1 KB)
  mlp_fold4_weights.pt  (577.1 KB)
  mlp_fold5_weights.pt  (577.1 KB)
  nn_timing_summary.csv  (0.4 KB)
  nn_timings.json  (2.4 KB)
  nn_train_test_metrics.csv  (0.7 KB)
  run_config.json  (0.8 KB)
  y_te_nn.pkl  (1165.0 KB)


## Βήμα 9: Train / validation / test metrics per fold, with TPR at 5% FPR

Reloads the best weights of every fold and scores the fold's training split, its validation split and the held-out test set. Nothing is retrained.

In a fresh session this step can be run without repeating the cross-validation: run Βήματα 0–4, skip Βήματα 5–8, then run this cell. It reads `cv_*_results.pkl` and the fold weights from `SAVE_DIR`.

In [ ]:
from sklearn.metrics import roc_curve


def load_cv_results(prefix):
    with open(os.path.join(SAVE_DIR, f"cv_{prefix}_results.pkl"), "rb") as f:
        results = pickle.load(f)
    for r in results:
        r["state_dict"] = torch.load(
            os.path.join(SAVE_DIR, f"{prefix}_fold{r['fold']}_weights.pt"), map_location="cpu")
    return results


if "cv_mlp_results" not in globals():
    cv_mlp_results = load_cv_results("mlp")
if "cv_ftt_results" not in globals():
    cv_ftt_results = load_cv_results("ftt")


def full_metrics(y, probs, thr=0.5):
    y = np.asarray(y).astype(int).ravel()
    tn, fp, fn, tp = confusion_matrix(y, (probs >= thr).astype(int), labels=[0, 1]).ravel()
    fpr, tpr, _ = roc_curve(y, probs)
    return {
        "N": len(y), "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
        "Accuracy": (tp + tn) / len(y),
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "Precision": tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        "F1-Score": 2 * tp / (2 * tp + fp + fn),
        "ROC-AUC": roc_auc_score(y, probs),
        "PR-AUC": average_precision_score(y, probs),
        "TPR@5%FPR": tpr[fpr <= 0.05].max(),
    }


def score_fold(name, res):
    train_idx, val_idx = FOLDS[res["fold"] - 1]
    d = prepare_fold(train_idx, val_idx)
    if name == "MLP":
        model = build_mlp(d["mlp_tr"].shape[1])
        sets = {"train": ([d["mlp_tr"]], d["y_tr"]),
                "val": ([d["mlp_val"]], d["y_val"]),
                "test": ([d["mlp_te"]], y_te_nn.values)}
        use_amp = False
    else:
        model = build_ftt(len(num_cols_nn))
        sets = {"train": ([d["num_tr"], d["cat_tr"]], d["y_tr"]),
                "val": ([d["num_val"], d["cat_val"]], d["y_val"]),
                "test": ([d["num_te"], d["cat_te"]], y_te_nn.values)}
        use_amp = (DEVICE.type == "cuda")
    model.load_state_dict(res["state_dict"])
    model = model.to(DEVICE).eval()

    rows = []
    for set_name, (arrays, y) in sets.items():
        inputs = [torch.as_tensor(a, device=DEVICE) for a in arrays]
        probs = torch.sigmoid(predict_logits(model, inputs, 8192, DEVICE, use_amp)).cpu().numpy().ravel()
        rows.append({"Model": name, "Fold": res["fold"], "Seed": res["seed"], "Set": set_name,
                     **full_metrics(y, probs)})
        del inputs
    # the test score recomputed here should match the one stored during the CV run
    drift = abs(rows[-1]["ROC-AUC"] - res["test_roc_auc"])
    print(f"  {name} fold {res['fold']}: train ROC-AUC {rows[0]['ROC-AUC']:.4f} | "
          f"test ROC-AUC {rows[-1]['ROC-AUC']:.4f} | TPR@5%FPR {rows[-1]['TPR@5%FPR']:.4f} "
          f"| diff vs stored {drift:.1e}")
    del model, d
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
    return rows


set_rows = []
for name, results in [("MLP", cv_mlp_results), ("FT-Transformer", cv_ftt_results)]:
    for res in results:
        set_rows += score_fold(name, res)

set_metrics = pd.DataFrame(set_rows)
set_cols = ["Accuracy", "Sensitivity", "Specificity", "Precision", "F1-Score",
            "ROC-AUC", "PR-AUC", "TPR@5%FPR"]
g = set_metrics.groupby(["Model", "Set"], sort=False)[set_cols]
set_summary = (g.mean().add_suffix(" mean").join(g.std(ddof=0).add_suffix(" std"))
               .reset_index())

set_metrics.to_csv(os.path.join(SAVE_DIR, "cv_train_val_test_metrics.csv"), index=False)
set_summary.to_csv(os.path.join(SAVE_DIR, "cv_train_val_test_summary.csv"), index=False)

print()
print(pd.DataFrame([
    {"Model": m, "Set": s, **{c: f"{grp[c].mean():.4f} +/- {grp[c].std(ddof=0):.4f}"
                             for c in ["ROC-AUC", "PR-AUC", "TPR@5%FPR", "Sensitivity", "Specificity"]}}
    for (m, s), grp in set_metrics.groupby(["Model", "Set"], sort=False)]).to_string(index=False))